# RAG Deep Dive — Employee Handbook

A single running example — an EY-style Employee Policy Handbook — used to walk through every stage of a real RAG pipeline: chunking strategies, embeddings, retrieval mechanics, reranking, and evaluation.

This notebook builds on the pipeline shape from **END TO END _ RAG.ipynb** and the chunking comparison from **CHUNKING STRATEGY DEMO.ipynb** — same handbook, expanded, taken all the way through the advanced techniques those notebooks didn't cover.

In [2]:
# %%capture
!pip install -qU langchain-groq langchain-openai openai pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 30.6 MB/s eta 0:00:00


In [4]:
!pip install -qU langchain-groq langgraph langchain-community faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
import os
from getpass import getpass
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
# Quick sanity check
print(llm.invoke("Say hello in exactly 3 words.").content)

Paste your Groq API key: ··········
Hello to you.


In [10]:
CLASSIFY_PROMPT = """Classify the user's question as exactly one word:
either "policy" or "general".
policy = questions about company rules, leave, expenses, equipment,
benefits, conduct, or HR processes.
general = anything else (small talk, general knowledge, unrelated topics).
Examples:
Question: How many paid leave days do I get per year?
Answer: policy
Question: What's a good recipe for banana bread?
Answer: general
Question: Can I expense a client dinner?
Answer: policy
Question: What is the capital of France?
Answer: general
Now classify this question. Answer with exactly one word, nothing else.
Question: {question}
Answer:"""
def classify_question(question: str) -> str:
    prompt = CLASSIFY_PROMPT.format(question=question)
    response = llm.invoke(prompt).content.strip().lower()
    # TODO: return "policy" if the word "policy" appears anywhere in the
    # response, else return "general"
    return "policy" if "policy" in response else "general"

# Test it -- both should print the correct category
print(classify_question("What is the laptop reimbursement policy?"))
print(classify_question("What's the weather like in Bangalore today?"))

policy
general


In [14]:
handbook_chunks = [
"""Leave Policy: Employees are entitled to 18 paid leave days per calendar
year, including casual and sick leave combined.""",
"""Laptop Policy: Company laptops are provided to all full-time employees
and must be returned upon exit. Personal use is permitted within
reasonable limits.""",
"""Remote Work Policy: Employees may work remotely up to 3 days per week
with manager approval, submitted via the HR portal.""",
"""Expense Policy: Business expenses including client meals and travel
are reimbursable with receipts submitted within 30 days.""",
"""Probation Policy: New employees undergo a 6-month probation period,
reviewed at the 3-month and 6-month marks.""",
"""Notice Period: Employees must serve a notice period of 60 days upon
resignation, unless otherwise agreed with HR.""",
"""Health Insurance: All employees are covered under group health
insurance from day one, extending to immediate family.""",
"""Working Hours: Standard working hours are 9:30 AM to 6:30 PM, Monday
to Friday, with flexible start times within a 1-hour window.""",
"""Grievance Redressal: Employees can raise workplace grievances
confidentially through the HR helpline, acknowledged within 2
working days.""",
"""Exit Process: Employees must complete a knowledge transfer plan
before their last working day; full settlement is processed
within 45 days.""",
]
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# TODO: build the FAISS index from handbook_chunks using embedding_model
vectorstore = FAISS.from_texts(handbook_chunks, embedding=embedding_model)
def retrieve(query: str, k: int = 2):
    # TODO: call the vectorstore's similarity search with `query` and `k`
    results = vectorstore.similarity_search(query, k=k)
    return [r.page_content for r in results]
# Test it
for chunk in retrieve("How many leave days do I get?"):
    print("-", chunk)

/tmp/ipykernel_782/528851134.py:26: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/tmp/ipykernel_782/528851134.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

- Leave Policy: Employees are entitled to 18 paid leave days per calendar
year, including casual and sick leave combined.
- Notice Period: Employees must serve a notice period of 60 days upon
resignation, unless otherwise agreed with HR.


In [17]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class HandbookState(TypedDict):
    question: str
    category: str
    context: str
    answer: str

def router_node(state: HandbookState):
    category = classify_question(state["question"])
    return {"category": category}

def rag_node(state: HandbookState):
    chunks = retrieve(state["question"], k=2)
    context = "\n".join(chunks)
    prompt = f"""Answer the question using ONLY the context below. Be concise.
Context:
{context}
Question: {state['question']}
Answer:"""
    # TODO: call the LLM with `prompt` and grab its .content
    answer = llm.invoke(prompt).content
    return {"context": context, "answer": answer}

def general_node(state: HandbookState):
    # TODO: call the LLM directly with the question (no retrieval needed here)
    answer = llm.invoke(state["question"]).content
    return {"context": "(no retrieval -- general question)", "answer": answer}

def route_decision(state: HandbookState) -> str:
    # TODO: return "rag_node" if the category is "policy", otherwise "general_node"
    return "rag_node" if state["category"] == "policy" else "general_node"

builder = StateGraph(HandbookState)
builder.add_node("router", router_node)
builder.add_node("rag_node", rag_node)
builder.add_node("general_node", general_node)
builder.add_edge(START, "router")
# TODO: add the conditional edge from "router" using route_decision,
# to the two possible destinations "rag_node" and "general_node"
builder.add_conditional_edges("router", route_decision, {"rag_node": "rag_node", "general_node": "general_node"})
builder.add_edge("rag_node", END)
builder.add_edge("general_node", END)
handbook_agent = builder.compile()
print("Graph compiled.")

# Test with a policy question -- what category and answer do you get?
result = handbook_agent.invoke({"question": "How many paid leave days do I get?"})
print("Category:", result["category"])
print("Answer:", result["answer"])
# Test with a general question -- what category and answer do you get?
result = handbook_agent.invoke({"question": "What's a fun fact about octopuses?"})
print("Category:", result["category"])
print("Answer:", result["answer"])

Graph compiled.
Category: policy
Answer: 18 days per calendar year.
Category: general
Answer: One fun fact about octopuses is that they have three hearts and blue blood. Two of the hearts are branchial hearts, which pump blood to the octopus's gills, while the third is a systemic heart that pumps blood to the rest of its body. The blue color of their blood comes from a copper-based molecule called hemocyanin, which is more efficient at transporting oxygen in cold, low-oxygen environments than the iron-based hemoglobin found in human blood. Isn't that cool?
